In [ ]:
pip install boto3 pandas

In [ ]:
import boto3
import os
import pandas as pd
import csv
import re

In [ ]:
# Initialize S3 client
aws_access_key='aws_access_key'
aws_secret_key='aws_secret_key'

# Initialize S3 client
s3 = boto3.client(service_name='s3',
        aws_access_key_id=aws_access_key,
        aws_secret_access_key=aws_secret_key,
        endpoint_url='endpoint_url')

In [ ]:
def list_s3_objects(bucket, prefix):
    """
    List all objects in the given S3 bucket with the given prefix (directory path).
    """
    objects = []
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get('Contents', []):
            objects.append(obj['Key'])
    return objects

In [ ]:
def load_fasttrackqc_csv(csv_path):
    """
    Load the fasttrackQC CSV into a pandas DataFrame
    """
    return pd.read_csv(csv_path, low_memory=False)

In [ ]:
def check_and_export_reconvert(s3_objects, prefix, fasttrack_df, reconvert_writer, delete_writer, processing_ready_writer):
    """
    Check files and export rows with discrepancies to reconvert.csv and delete_files_from_s3.csv
    """
    # Extract unique subjects from s3_objects
    subjects = set(f.split('/')[2] for f in s3_objects if prefix in f)
    print(subjects)
    
    
    for subject in subjects:
        # Loop through sessions for each subject
        sessions = set(f.split('/')[3] for f in s3_objects if f.startswith(f'{prefix}/{subject}/'))
        
        for session in sessions:
            if subject == 'sourcedata' or '.tsv' in session:
                continue
            print(f"Processing subject: {subject}, session: {session}")
            bids_year=session
            print(bids_year)
            
            processing_ready = 1
            
            # Initialize lists for modalities
            modalities = ['anat', 'dwi', 'func', 'fmap']
            anat_files = [f for f in s3_objects if f.startswith(f'{prefix}/{subject}/{session}/anat')]

            # Check T1W and T2W in anat
            t1w_files = [f for f in anat_files if f'{subject}_{session}_T1w.nii.gz' in f or f'{subject}_{session}_rec-normalized_T1w.nii.gz' in f or re.search(r'rec-normalized_run-\d+_T1w.nii.gz', f) or re.search(rf'{subject}_{session}_run-\d+_T1w.nii.gz', f)]
            t2w_files = [f for f in anat_files if f'{subject}_{session}_T2w.nii.gz' in f or f'{subject}_{session}_rec-normalized_T2w.nii.gz' in f or re.search(r'rec-normalized_run-\d+_T2w.nii.gz', f) or re.search(rf'{subject}_{session}_run-\d+_T2w.nii.gz', f)]

            # Determine T1 series type based on the filenames present
            if any(f'{subject}_{session}_T1w' in f or re.search(rf'{subject}_{session}_run-\d+_T1w.nii.gz', f) for f in t1w_files):
                print("Setting t1w_series_type: ABCD-T1")
                t1w_series_type = 'ABCD-T1'
            elif any(f'{subject}_{session}_rec-normalized_T1w' in f or re.search(rf'{subject}_{session}_rec-normalized_run-\d+_T1w.nii.gz', f) for f in t1w_files):
                print("Setting t1w_series_type: ABCD-T1-NORM")
                t1w_series_type = 'ABCD-T1-NORM'
            else:
                print("No ant files found, reconvert")
                t1w_series_type='none'
                processing_ready=0
                
            # Determine T2 series type based on the filenames present
            if any(f'{subject}_{session}_T2w' in f or re.search(rf'{subject}_{session}_run-\d+_T2w.nii.gz', f) for f in t2w_files):
                t2w_series_type = 'ABCD-T2'
            elif any(f'{subject}_{session}_rec-normalized_T2w' in f or re.search(rf'{subject}_{session}_rec-normalized_run-\d+_T2w.nii.gz', f) for f in t2w_files):
                t2w_series_type = 'ABCD-T2-NORM'
            else:
                print("No ant T2 files found, reconvert")
                t2w_series_type='none'
                                                                                                              
            # Check the count of T1W and T2W files in fasttrackQC
            t1w_fasttrackqc_count = len(fasttrack_df[(fasttrack_df['SeriesType'] == t1w_series_type) & 
                                                     (fasttrack_df['usable'] != 0) & 
                                                     (fasttrack_df['pGUID'].str.contains(subject.split('-')[1], na=False)) & 
                                                     (fasttrack_df['EventName'] == bids_year)])

            t2w_fasttrackqc_count = len(fasttrack_df[(fasttrack_df['SeriesType'] == t2w_series_type) & 
                                                     (fasttrack_df['usable'] != 0) & 
                                                     (fasttrack_df['pGUID'].str.contains(subject.split('-')[1], na=False)) & 
                                                     (fasttrack_df['EventName'] == bids_year)])
            print(f"t1w_fasttrackqc_count: {t1w_fasttrackqc_count} ")
            print(f"t2w_fasttrackqc_count: {t2w_fasttrackqc_count} ")

            # Get S3 counts for T1W and T2W
            s3_t1w_count = len(t1w_files)
            s3_t2w_count = len(t2w_files)
            print(f"s3_t1w_count: {s3_t1w_count} ")
            print(f"s3_t2w_count: {s3_t2w_count} ")

            # Compare T1W counts and handle discrepancies
            if t1w_fasttrackqc_count < s3_t1w_count:
                print(f"t1w_fasttrackqc_count ({t1w_fasttrackqc_count}) < s3_t1w_count ({s3_t1w_count})")
                # More T1W files in S3 than in fasttrackQC -> log to delete_files_from_s3.csv
                delete_writer.writerow([subject, session,'t1w', t1w_files])
                processing_ready=0
            
            if t1w_fasttrackqc_count > s3_t1w_count:
                print(f"t1w_fasttrackqc_count ({t1w_fasttrackqc_count}) > s3_t1w_count ({s3_t1w_count})")
                # More T1W files in fasttrackQC than in S3 -> log to reconvert.csv
                reconvert_writer.writerow([subject, session, 'More T1W files in fasttrackQC than in S3'])
                processing_ready=0
                
            # Compare T2W counts and handle discrepancies
            if t2w_fasttrackqc_count < s3_t2w_count:
                print(f"t2w_fasttrackqc_count ({t2w_fasttrackqc_count}) < s3_t2w_count ({s3_t2w_count})")
                # More T2W files in S3 than in fasttrackQC -> log to delete_files_from_s3.csv
                delete_writer.writerow([subject, session,'t2w', t2w_files])
                processing_ready=0
            
            if t2w_fasttrackqc_count > s3_t2w_count:
                print(f"t2w_fasttrackqc_count ({t2w_fasttrackqc_count}) > s3_t2w_count ({s3_t2w_count})")
                # More T2W files in fasttrackQC than in S3 -> log to reconvert.csv
                reconvert_writer.writerow([subject, session, 'More T2W files in fasttrackQC than in S3'])
                processing_ready=0

            # Load dwi_valid CSV
            dwi_valid_df = pd.read_csv('/path/to/valid_dwi_cases.csv') #csv file with participants expected to have fucntional derivatives 
            dwi_fmap_check=1
            # Check if subject and session are in dwi_valid
            if not dwi_valid_df[(dwi_valid_df['subject'] == subject) & (dwi_valid_df['session'] == session)].empty:
                print(f"Subject {subject} and session {session} are valid for DWI checks.") 
                # Check DWI files in dwi directory
                dwi_files = [f for f in s3_objects if f.startswith(f'{prefix}/{subject}/{session}/dwi')]
                s3_dwi_files = [f for f in dwi_files if '_dwi.nii.gz' in f]

                # Count number of valid dMRI entries in fasttrackQC
                dwi_fasttrackqc_count = len(fasttrack_df[(fasttrack_df['SeriesType'] == 'ABCD-DTI') & 
                                                         (fasttrack_df['usable'] != 0) & 
                                                         (fasttrack_df['pGUID'].str.contains(subject.split('-')[1], na=False)) & 
                                                         (fasttrack_df['EventName'] == bids_year)])

                # Compare DWI counts and handle discrepancies
                s3_dwi_count = len(s3_dwi_files)
                print("dwi_fasttrackqc_count:", dwi_fasttrackqc_count)
                print("s3_dwi_count", s3_dwi_count)

                if dwi_fasttrackqc_count < s3_dwi_count:
                    # More DWI files in S3 than in fasttrackQC -> log to delete_files_from_s3.csv
                    print(f"dwi_fasttrackqc_count ({dwi_fasttrackqc_count}) < s3_dwi_count ({s3_dwi_count})")
                    delete_writer.writerow([subject, session,'dwi', s3_dwi_files])
                    processing_ready=0


                if dwi_fasttrackqc_count > s3_dwi_count:
                    # More DWI files in fasttrackQC than in S3 -> log to reconvert.csv
                    print(f"dwi_fasttrackqc_count ({dwi_fasttrackqc_count}) > s3_dwi_count ({s3_dwi_count})")
                    reconvert_writer.writerow([subject, session, 'More DWI files in fasttrackQC than in S3'])
                    processing_ready=0
            else:
                dwi_fmap_check=0
                print(f"Subject {subject} and session {session} are not valid for DWI checks. Skipping.")
                
            # Load func_valid CSV
            func_valid_df = pd.read_csv('/path/to/valid_func_cases.csv') #csv file with participants expected to have fucntional derivatives 
            func_fmap_check=1
            if not func_valid_df[(func_valid_df['subject'] == subject) & (func_valid_df['session'] == session)].empty:
                print(f"Subject {subject} and session {session} are valid for func checks.") 
                # Define a list of task names
                tasks = ['MID', 'nback', 'SST', 'rest']
                # Check task files in func directory
                func_files = [f for f in s3_objects if f.startswith(f'{prefix}/{subject}/{session}/func')]
                #Check for ERI data
                eri_files = [f for f in s3_objects if f.startswith(f'{prefix}/sourcedata/{subject}/{session}/func')]

                # Loop over each task
                for task in tasks:

                    # Construct the file name pattern based on the task
                    s3_task_files = [f for f in func_files if f'{subject}_{session}_task-{task}_bold.nii.gz' in f or re.search(rf'task-{task}_run-\d+_bold.nii.gz', f)]
                    if task != 'rest':
                        s3_eri_files = [f for f in eri_files if f'{subject}_{session}_task-{task}_bold_EventRelatedInformation.txt' in f or re.search(rf'task-{task}_run-\d+_bold_EventRelatedInformation.txt', f)]

                    #Set task SeriesType on the basis of task
                    if task == 'MID':
                        task_series_type = 'ABCD-MID-fMRI'
                    elif task == 'nback':
                        task_series_type = 'ABCD-nBack-fMRI'
                    elif task == 'SST':
                        task_series_type = 'ABCD-SST-fMRI'
                    else: task_series_type = 'ABCD-rsfMRI'

                    # Count number of valid task entries in fasttrackQC
                    task_fasttrackqc_count = len(fasttrack_df[(fasttrack_df['SeriesType'] == task_series_type) & 
                                                              (fasttrack_df['usable'] != 0) & 
                                                              (fasttrack_df['pGUID'].str.contains(subject.split('-')[1], na=False)) & 
                                                              (fasttrack_df['EventName'] == bids_year)])

                    # Compare task counts and handle discrepancies
                    s3_task_count = len(s3_task_files)

                    print(f"{task}_fasttrackqc_count:", task_fasttrackqc_count)
                    print(f"{task}_s3_task_count", s3_task_count)
                    if task != 'rest':
                        s3_eri_count = len(s3_eri_files)
                        print(f"{task}_s3_eri_count", s3_eri_count)

                    if task_fasttrackqc_count < s3_task_count:
                        # More task files in S3 than in fasttrackQC -> log to delete_files_from_s3.csv
                        print(f"{task}_fasttrackqc_count ({task_fasttrackqc_count}) < s3_{task}_count ({s3_task_count})")
                        extra_task_files = s3_task_files
                        delete_writer.writerow([subject, session, task, extra_task_files])
                        processing_ready=0

                    if task_fasttrackqc_count > s3_task_count:
                        print(f"{task}_fasttrackqc_count ({task_fasttrackqc_count}) > s3_{task}_count ({s3_task_count})")
                        # More task files in fasttrackQC than in S3 -> log to reconvert.csv
                        reconvert_writer.writerow([subject, session, f'More {task} files in fasttrackQC than in S3'])
                        processing_ready=0

                    if task != 'rest':
                        if task_fasttrackqc_count < s3_eri_count:
                            # More task files in S3 than in fasttrackQC -> log to delete_files_from_s3.csv
                            print(f"{task}_fasttrackqc_count ({task_fasttrackqc_count}) < s3_{task}_eri_count ({s3_eri_count})")
                            extra_task_files = s3_task_files
#                             delete_writer.writerow([subject, session, task, extra_task_files])
                        elif task_fasttrackqc_count > s3_eri_count:
                            print(f"{task}_fasttrackqc_count ({task_fasttrackqc_count}) > s3_{task}_eri_count ({s3_eri_count})")
                            # More task files in fasttrackQC than in S3 -> log to reconvert.csv
#                             reconvert_writer.writerow([subject, session, f'More {task} eri files in fasttrackQC than in S3'])
                                                      
            else:
                func_fmap_check=0
                print(f"Subject {subject} and session {session} are not valid for func checks. Skipping.")                                    
             
                                                 
            # Check task files in func directory
            fmap_files = [f for f in s3_objects if f.startswith(f'{prefix}/{subject}/{session}/fmap')]

            # Initialize counters for func and dwi AP and PA fmap counts
            s3_fmap_counts = {"func": {"AP": 0, "PA": 0}, "dwi": {"AP": 0, "PA": 0}}

            # Loop over dwi and func fmaps
            for acq_type in ['func', 'dwi']:
                # Skip checks based on flags
                if acq_type == 'func' and func_fmap_check == 0:
                    print("Skipping func checks as func_fmap_check = 0")
                    continue
                if acq_type == 'dwi' and dwi_fmap_check == 0:
                    print("Skipping dwi checks as dwi_fmap_check = 0")
                    continue

                # Loop over each fmap direction (AP, PA)
                for direc in ['AP', 'PA']:
                    # Construct the file name pattern based on the task
                    s3_fmap_files = [
                        f for f in fmap_files
                        if f'{subject}_{session}_acq-{acq_type}_dir-{direc}_epi.nii.gz' in f or
                           re.search(rf'_acq-{acq_type}_dir-{direc}_run-\d+_epi.nii.gz', f)
                    ]

                    if acq_type == 'func':
                        seriesType = ['ABCD-fMRI-FM', f'ABCD-fMRI-FM-{direc}']
                    else:
                        seriesType = ['ABCD-Diffusion-FM', f'ABCD-Diffusion-FM-{direc}']

                    # Count number of valid fmap entries in fasttrackQC
                    fmap_fasttrackqc_count = len(fasttrack_df[
                        (fasttrack_df['SeriesType'].isin(seriesType)) &
                        (fasttrack_df['usable'] != 0) &
                        (fasttrack_df['pGUID'].str.contains(subject.split('-')[1], na=False)) &
                        (fasttrack_df['EventName'] == bids_year)
                    ])

                    # Compare task counts and handle discrepancies
                    s3_fmap_count = len(s3_fmap_files)
                    s3_fmap_counts[acq_type][direc] = s3_fmap_count

                    print(f"{acq_type}_{direc}_fasttrackqc_count:", fmap_fasttrackqc_count)
                    print(f"{acq_type}_{direc}_s3_fmap_count:", s3_fmap_count)

                    if fmap_fasttrackqc_count < s3_fmap_count:
                        # More fmap files in S3 than in fasttrackQC -> log to delete_files_from_s3.csv
                        print(f"{acq_type}_{direc}_fasttrackqc_count ({fmap_fasttrackqc_count}) < s3_{acq_type}_{direc}_count ({s3_fmap_count})")
                        extra_task_files = s3_fmap_files
                        delete_writer.writerow([subject, session, direc, extra_task_files])
                        processing_ready = 0

                    elif fmap_fasttrackqc_count > s3_fmap_count:
                        if acq_type == 'dwi':
                            print(f"{acq_type}_{direc}_fasttrackqc_count ({fmap_fasttrackqc_count}) > s3_{acq_type}_{direc}_count ({s3_fmap_count}): Check manufacturer")
                        else:
                            print(f"{acq_type}_{direc}_fasttrackqc_count ({fmap_fasttrackqc_count}) > s3_{acq_type}_{direc}_count ({s3_fmap_count})")

            # Check if func_AP == func_PA and dwi_AP == dwi_PA to avoid writing to reconvert
            for acq_type in ['func']:
                if s3_fmap_counts[acq_type]['AP'] != s3_fmap_counts[acq_type]['PA']:
                    if s3_fmap_counts[acq_type]['AP'] > s3_fmap_counts[acq_type]['PA']:
                        direc='AP'
                        reconvert_writer.writerow([
                            subject,
                            session,
                            f"More {acq_type}_{direc} files in fasttrackQC than in S3"
                        ])
                    else: 
                        direc='AP'
                        reconvert_writer.writerow([
                            subject,
                            session,
                            f"More {acq_type}_{direc} files in fasttrackQC than in S3"
                        ])
                    processing_ready = 0
                else:
                    print(f"Counts match for {acq_type}_AP and {acq_type}_PA; skipping reconvert write.")
                        
            #Export to csv to hit go on abcd-hcp processing pipeline
            if processing_ready:
                processing_ready_writer.writerow([subject, session])

In [ ]:
def main(bucket_name, prefix, fasttrack_csv_path, reconvert_csv_path, delete_csv_path, processing_ready_csv_path):
    # Load fasttrackQC.csv
    fasttrack_df = load_fasttrackqc_csv(fasttrack_csv_path)

    # Create CSV files for reconvert and delete
    with open(reconvert_csv_path, mode='w', newline='') as reconvert_file, \
         open(processing_ready_csv_path, mode='w', newline='') as processing_ready_file, \
         open(delete_csv_path, mode='w', newline='') as delete_file:
        
        reconvert_writer = csv.writer(reconvert_file)
        processing_ready_writer = csv.writer(processing_ready_file)
        delete_writer = csv.writer(delete_file)

        reconvert_writer.writerow(['Subject', 'Session', 'Issue'])
        delete_writer.writerow(['Subject', 'Session','Task', 'File to Delete'])
        processing_ready_writer.writerow(['Subject', 'Session'])

        # List all objects in S3 bucket
        s3_objects = list_s3_objects(bucket_name, prefix)
        print(s3_objects)
        
        # Check and export reconvert and delete issues
        check_and_export_reconvert(s3_objects,prefix, fasttrack_df, reconvert_writer, delete_writer, processing_ready_writer)

    print(f"Reconvert file generated at: {reconvert_csv_path}")
    print(f"Delete files list generated at: {delete_csv_path}")
    print(f"Processing ready list generated at: {processing_ready_csv_path}")

if __name__ == '__main__':
    # Configuration
    bucket_name = 'bucket_name'
    prefix = 'path/to/bids/data/' # path after the bucket_name where you bids data to be audited lives 
    fasttrack_csv_path = '/path/to/fasttrack_mri_qc.csv'  # QC file provided by DAIRC
    reconvert_csv_path = '/path/to/audit/dir/reconvert.csv'   # csv file where cases to be reconverted are reported 
    delete_csv_path = '/path/to/audit/dir/delete_files_from.csv' # csv file with cases which have extra files/ need to be deleted
    processing_ready_csv_path ='/path/to/audit/dir/processing_ready.csv' # csv file with cases ready for further processing
    
    main(bucket_name, prefix, fasttrack_csv_path, reconvert_csv_path, delete_csv_path, processing_ready_csv_path)

## Delete the subjects in delete file and then rerun bids2dicom 

In [ ]:
import csv
import subprocess

# Set AWS credentials as environment variables
os.environ['AWS_ACCESS_KEY_ID'] = 'aws_access_key'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'aws_secret_access_key'

# Define the input and output CSV file paths
input_csv = '/path/to/inptu/csv'  # Input CSV with subject and session columns
output_csv = '/path/to/output/csv'  # Output CSV to store results

# A set to track which subject/session pairs have already been processed
processed_subjects_sessions = set()

# Define S3 bucket base URL
s3_bucket_base = 'bucket_name'

def delete_from_s3(subject, session):
    """Runs s3cmd to delete a directory from S3."""
    s3_path = f"{s3_bucket_base}/{subject}/{session}"
    try:
        # Execute the s3cmd delete command
        subprocess.run(['s3cmd', 'del', '-r', s3_path], check=True)
        return True
    except subprocess.CalledProcessError as e:
        print(f"Error deleting {s3_path}: {e}")
        return False

def process_csv(input_file, output_file):
    """Reads input CSV, processes the subjects/sessions, and writes results to output CSV."""
    # Read the input CSV file
    with open(input_file, mode='r') as infile:
        reader = csv.DictReader(infile)
        
        # Open output CSV to write the result
        with open(output_file, mode='w', newline='') as outfile:
            fieldnames = ['subject', 'session']
            writer = csv.DictWriter(outfile, fieldnames=fieldnames)
            writer.writeheader()

            # Process each row in the input CSV
            for row in reader:
                subject = row['Subject']
                session = row['Session']
                subject_session_pair = (subject, session)

                # If the subject/session pair has already been processed, skip it
                if subject_session_pair in processed_subjects_sessions:
                    continue  # Skip this entry, as it has already been processed

                # Attempt to delete from S3
                deleted = delete_from_s3(subject, session)

                # If deletion was successful, mark this pair as processed
                if deleted:
                    processed_subjects_sessions.add(subject_session_pair)

            # Write the processed subject/session pairs to the output CSV
            for subject, session in processed_subjects_sessions:
                writer.writerow({
                    'subject': subject,
                    'session': session
                })

if __name__ == '__main__':
    # Run the CSV processing
    process_csv(input_csv, output_csv)
    print(f"Process complete. Results saved in {output_csv}")